In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Carregar dataset espelhado
df = pd.read_csv('../data/annotated/annotations_mirrored_batch.csv')

# Separar EN e PT
en = df[df['language'] == 'EN'].copy()
pt = df[df['language'] == 'PT'].copy()

print(f"Pares EN: {len(en)}")
print(f"Pares PT: {len(pt)}")
print(f"\nColunas: {df.columns.tolist()}")

In [ ]:
#Verificar alinhamento dos pares

# Extrair o ID base (sem -EN ou -PT)
en['base_id'] = en['id'].str.replace('-EN', '')
pt['base_id'] = pt['id'].str.replace('-PT', '')

# Ordenar por base_id para garantir alinhamento
en = en.sort_values('base_id').reset_index(drop=True)
pt = pt.sort_values('base_id').reset_index(drop=True)

# Verificar alinhamento
print("Pares alinhados:")
for i in range(len(en)):
    print(f"  {en.loc[i,'base_id']}: EN={en.loc[i,'id']} | PT={pt.loc[i,'id']}")

In [ ]:
#Calcular kappa por dimensão

dimensoes = ['faithfulness', 'relevance', 'fluency', 'completeness', 'safety']
resultados = {}

print("=" * 55)
print(f"{'Dimensão':<15} {'Kappa Simples':>14} {'Kappa Ponderado':>16} {'Status':>8}")
print("=" * 55)

for dim in dimensoes:
    scores_en = en[dim].tolist()
    scores_pt = pt[dim].tolist()
    
    # Verificar variação
    if len(set(scores_en + scores_pt)) == 1:
        # Sem variação — todos scores iguais
        print(f"{dim:<15} {'N/A (sem variação)':>30}  ⚠️")
        resultados[dim] = {'kappa': None, 'kappa_w': None, 'status': 'sem_variacao'}
    else:
        try:
            kappa = cohen_kappa_score(scores_en, scores_pt)
            kappa_w = cohen_kappa_score(scores_en, scores_pt, weights='quadratic')
            status = "✅" if kappa_w >= 0.60 else "⚠️"
            print(f"{dim:<15} {kappa:>14.3f} {kappa_w:>16.3f} {status:>8}")
            resultados[dim] = {'kappa': kappa, 'kappa_w': kappa_w, 'status': 'ok'}
        except Exception as e:
            print(f"{dim:<15} {'Erro: ' + str(e):>30}")
            resultados[dim] = {'kappa': None, 'kappa_w': None, 'status': 'erro'}

print("=" * 55)

In [ ]:
#Kappa para halucination

print("\nKappa para hallucination_type (categórico):")
print("-" * 40)

hal_en = en['hallucination_type'].tolist()
hal_pt = pt['hallucination_type'].tolist()

if len(set(hal_en + hal_pt)) == 1:
    print(f"Todos os pares classificados como '{hal_en[0]}' em EN e PT.")
    print("Kappa não calculável — sem variação nas categorias.")
    print("→ Interpretação: concordância perfeita por construção.")
else:
    kappa_hal = cohen_kappa_score(hal_en, hal_pt)
    print(f"Cohen's Kappa: {kappa_hal:.3f}")